# Agentic AI & RAG Engineering — Course Notebook (Weeks 1–3)

**Author:** Narayanan Palani  
**Scope:** Weeks 1 to 3 Complete Implementation Reference  
**Stack:** Python, Ollama (`gpt-oss:20b`), Pydantic v2, AsyncIO, SQLite, FastAPI, Streamlit, Pytest

--- 
## 1. Environment Setup & Dependency Configuration
* **Slide Source:** *Week 1 - Slide 17 ("Setup — Two Minutes"), Week 2 - Slide 26 ("Secrets — Extending the W1 Discipline")*
* **Action:** Install required packages and securely load API credentials using `python-dotenv`.

In [8]:
# Week 1 - Hello LLM
# Purpose: Run a first local LLM request through Ollama.
#
# One-time setup:
#   1. Install Ollama from https://ollama.com using code at cmd prompt: curl -fsSL https://ollama.com/install.sh | sh
#   2. Start the Ollama service in an exclusive command prompt: ollama run gpt-oss:20b
#   3. Pull the local model from Terminal:
#        ollama pull gpt-oss:20b
#
# Ollama runs the model locally, so this inference does not use OpenAI API credits
# or require an OPENAI_API_KEY.

# Install the Ollama Python package in this notebook environment.
# Run this once if the package is not already installed.
%pip install -q ollama

# Import the Ollama Python client.
import ollama

# Send a chat request to the local Ollama model.
response = ollama.chat(
    # Use the same local model throughout this notebook.
    model="gpt-oss:20b",

    # Send only the user message needed for this experiment.
    messages=[
        {
            "role": "user",
            "content": "What is the core benefit of RAG? Answer with exactly one word."
        }
    ]
)

# Print the text generated by the local model.
print(response.message.content)


Note: you may need to restart the kernel to use updated packages.


HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


Accuracy


In [9]:
# Slide Source:
#   Week 1 - Slide 17 ("Setup — Two Minutes")
#   Week 2 - Slide 26 ("Secrets — Extending the W1 Discipline")
#
# Purpose:
#   1. Verify that the Ollama Python package is available.
#   2. Confirm that the local Ollama service is reachable.
#   3. Confirm that the selected local model is installed.
#
# Local-model principle:
#   Ollama runs inference on your machine.
#   No OPENAI_API_KEY is required for these exercises.
#   No OpenAI API credits are consumed by local inference calls.
#
# Before running this cell:
#   1. Install Ollama from https://ollama.com
#   2. Start the Ollama service.
#   3. Pull the model from Terminal:
#        ollama pull gpt-oss:20b

import ollama

# Query the local Ollama service for installed models.
# This checks connectivity without generating a model response.
models = ollama.list()

# Extract the local model names.
installed_models = [model.model for model in models.models]

# Stop early with a clear instruction if the required model is missing.
if not any(name == "gpt-oss:20b" for name in installed_models):
    raise RuntimeError(
        f"Local Ollama model 'gpt-oss:20b' was not found. "
        f"Run: ollama pull gpt-oss:20b"
    )

# Confirm that the local environment is ready.
print("Local Ollama environment successfully initialized.")
print(f"Model available: gpt-oss:20b")


HTTP Request: GET http://127.0.0.1:11434/api/tags "HTTP/1.1 200 OK"


Local Ollama environment successfully initialized.
Model available: gpt-oss:20b


--- 
## 2. Week 1: Foundations, Decision Frameworks & Hello LLM
* **Slide Source:** *Week 1 - Slide 06 ("A Working Definition"), Slide 11-16 ("Four Patterns"), Slide 17-18 ("Hello LLM & Lab Step 2")*
* **Action:** Initialize a local Ollama client, run a baseline prompt using `gpt-oss:20b`, and log local inference metrics.

In [10]:
# Local Ollama connectivity check
#
# This replaces a cloud-model/API connectivity check.
# It verifies that:
#   1. The Ollama service is running locally.
#   2. The Python client can communicate with it.
#   3. Local models are visible to the client.
#
# No LLM prompt is generated by this check.

import ollama

# Ask the local Ollama service for its installed model list.
models = ollama.list()

print("Ollama service is reachable.")
print("Installed local models:")

# Display the local models available to this notebook.
for model in models.models:
    print(f"- {model.model}")


HTTP Request: GET http://127.0.0.1:11434/api/tags "HTTP/1.1 200 OK"


Ollama service is reachable.
Installed local models:
- gpt-oss:20b


In [2]:
# Week 1 - Hello LLM / Lab Step 2
#
# This version uses Ollama instead of the local Ollama service.
# The model runs locally, so there is no API key or API credit requirement.

# Import the Ollama Python client.
import ollama

# Use one model consistently throughout the notebook.
OLLAMA_MODEL = "gpt-oss:20b"


def run_hello_llm(prompt_text: str) -> str:
    """
    Send a prompt to the local Ollama model and return its response.

    prompt_text:
        The question/instruction sent to the local model.

    Returns:
        The text generated by the local model.
    """

    # Send the prompt to Ollama running on the local machine.
    response = ollama.chat(
        model=OLLAMA_MODEL,
        messages=[
            {
                "role": "user",
                "content": prompt_text
            }
        ],
        options={
            # Keep generation focused and relatively deterministic.
            "temperature": 0.2,
            # Limit generation for this one-word experiment.
            "num_predict": 1000
        }
    )

    # Ollama reports local inference statistics in the response.
    # prompt_eval_count = input tokens evaluated locally.
    # eval_count = output tokens generated locally.
    prompt_tokens = getattr(response, "prompt_eval_count", 0)
    completion_tokens = getattr(response, "eval_count", 0)

    # Calculate total locally processed tokens for visibility.
    total_tokens = prompt_tokens + completion_tokens

    print(
        f"Prompt tokens: {prompt_tokens} | "
        f"Completion tokens: {completion_tokens} | "
        f"Total tokens: {total_tokens}"
    )

    # Extract and return only the generated text.
    return response.message.content


# Keep the prompt short to demonstrate prompt-efficiency principles.
prompt = "What is the core benefit of RAG? Answer with exactly one word."

# Run the local model.
result = run_hello_llm(prompt)

# Display the generated response.
print("\nLLM Response:")
print(result)


Prompt tokens: 82 | Completion tokens: 423 | Total tokens: 505

LLM Response:
Accuracy


--- 
## 3. Week 2: Typed Contracts (Pydantic) & Async Concurrency Pipeline
* **Slide Source:** *Week 2 - Slide 05-07 ("Pydantic Models"), Slide 09-11 ("Async Basics & httpx"), Slide 15-18 ("Concurrency Patterns: Gather, Batching, Retry"), Slide 23 ("Structured Logging"), Slide 27 ("SQLite Store")*
* **Action:** Build Pydantic schemas, an async client with exponential retry backoff, parallel batch processing via `asyncio.gather`, JSON structured logging, and SQLite persistence.

In [12]:
# Slide Source: Week 2 - Slide 05-07 ("Pydantic Models"), Slide 09-11 ("Async Basics & httpx"),
# Slide 15-18 ("Concurrency Patterns: Gather, Batching, Retry"), Slide 23 ("Structured Logging"), Slide 27 ("SQLite Store")
#
# Action: Build Pydantic schemas, an async Ollama client with exponential retry
# backoff, parallel batch processing via asyncio.gather, JSON structured logging,
# and SQLite persistence.
#
# Ollama replaces the paid cloud API for this notebook. The model runs locally.

import asyncio
import json
import logging
import sqlite3
import time
from typing import List
from ollama import AsyncClient
from pydantic import BaseModel, Field

# Use the same local model throughout the notebook.
OLLAMA_MODEL = "gpt-oss:20b"

# Global lock to serialize database writes across concurrent tasks
db_lock = asyncio.Lock()

# Structured Logging Setup
logging.basicConfig(level=logging.INFO, format="%(message)s")


def log_json(event: str, **kwargs):
    log_entry = {"event": event, "timestamp": time.time(), **kwargs}
    logging.info(json.dumps(log_entry))


# Pydantic Schemas
class QuestionRequest(BaseModel):
    id: int
    query: str = Field(..., min_length=3, description="The user query")


class AnswerResponse(BaseModel):
    id: int
    query: str
    answer: str
    status: str = "success"


# SQLite Persistence Initialization
def init_db():
    # Use timeout to wait up to 20 seconds for lock release if database is busy
    conn = sqlite3.connect("results.db", timeout=20.0)
    cursor = conn.cursor()

    # Enable Write-Ahead Logging (WAL) mode for drastically better concurrency
    cursor.execute("PRAGMA journal_mode=WAL;")

    cursor.execute(
        """
        CREATE TABLE IF NOT EXISTS run_results (
            id INTEGER PRIMARY KEY,
            query TEXT NOT NULL,
            answer TEXT NOT NULL,
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        )
    """
    )

    conn.commit()
    conn.close()


def _sync_save_to_db(record: AnswerResponse):
    """Synchronous SQLite write operation with extended timeout."""
    conn = sqlite3.connect("results.db", timeout=20.0)
    cursor = conn.cursor()

    cursor.execute(
        "INSERT INTO run_results (id, query, answer) VALUES (?, ?, ?)",
        (record.id, record.query, record.answer),
    )

    conn.commit()
    conn.close()


async def save_to_db(record: AnswerResponse):
    """Thread-safe, non-blocking async wrapper around DB writes."""
    async with db_lock:
        # Offload blocking SQLite I/O to a worker thread
        await asyncio.to_thread(_sync_save_to_db, record)


# Retry Wrapper with Exponential Backoff
async def with_retry(coro_func, *args, max_retries: int = 3, **kwargs):
    for attempt in range(1, max_retries + 1):
        try:
            return await coro_func(*args, **kwargs)
        except Exception as exc:
            log_json(
                "async_retry_attempt",
                attempt=attempt,
                max_retries=max_retries,
                error=str(exc),
            )

            if attempt == max_retries:
                raise

            await asyncio.sleep(2**attempt)


async def process_single_query(
    async_client: AsyncClient, req: QuestionRequest
) -> AnswerResponse:
    async def _call():
        response = await async_client.chat(
            model=OLLAMA_MODEL,
            messages=[{"role": "user", "content": req.query}],
            options={"temperature": 0.3},
        )

        # Extract message content safely
        if hasattr(response, "message"):
            return response.message.content.strip()
        return response["message"]["content"].strip()

    answer_text = await with_retry(_call)

    res = AnswerResponse(id=req.id, query=req.query, answer=answer_text)

    # Safely save to DB asynchronously without lock contention
    await save_to_db(res)

    log_json("query_processed", id=req.id, query=req.query)
    return res


async def batch_process_pipeline(queries: List[QuestionRequest]):
    init_db()
    async_client = AsyncClient()

    tasks = [process_single_query(async_client, q) for q in queries]

    results = await asyncio.gather(*tasks, return_exceptions=True)
    return results


# Driver Code Execution
async def main():
    sample_queries = [
        QuestionRequest(id=1, query="What is Pydantic in Python?"),
        QuestionRequest(
            id=2, query="How does asyncio.gather enable concurrent local model calls?"
        ),
        QuestionRequest(
            id=3, query="Why use SQLite for local execution persistence?"
        ),
    ]

    print("Starting Async Local Ollama Batch Pipeline...")
    batch_results = await batch_process_pipeline(sample_queries)

    for res in batch_results:
        if isinstance(res, Exception):
            print(f"\nBatch item failed: {res}")
        else:
            print(
                f"\nID: {res.id}"
                f"\nQuery: {res.query}"
                f"\nAnswer: {res.answer[:100]}..."
            )


if __name__ == "__main__":
    if "get_ipython" in globals():
        import nest_asyncio

        nest_asyncio.apply()
        asyncio.run(main())
    else:
        asyncio.run(main())

Starting Async Local Ollama Batch Pipeline...


HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"



Batch item failed: database is locked

Batch item failed: database is locked

Batch item failed: database is locked


--- 
## 4. Week 3: FastAPI Web Service & Streaming Endpoint
* **Slide Source:** *Week 3 - Slide 06 ("FastAPI 12-line App"), Slide 08 ("/health Endpoint"), Slide 12-14 ("Streaming via StreamingResponse")*
* **Action:** Write `main.py` containing FastAPI backend supporting a health probe, structured POST endpoint, and streaming token response via SSE / raw stream.

In [1]:
%%writefile main.py
# Slide Source: Week 3 - Slide 06 ("FastAPI 12-line App"), Slide 08 ("/health Endpoint"), Slide 12-14 ("Streaming via StreamingResponse")
#
# Action: Build FastAPI backend supporting a health probe, a structured POST
# endpoint, and a streaming response backed by a local Ollama model.
#
# Run locally with:
#   uvicorn main:app --reload --port 8000

from fastapi import FastAPI, HTTPException
from fastapi.responses import StreamingResponse
from pydantic import BaseModel
from ollama import AsyncClient

# The local Ollama model used by this backend.
OLLAMA_MODEL = "gpt-oss:20b"

# Create the FastAPI application.
app = FastAPI(
    title="Agentic RAG Engine API",
    version="2.0.0"
)

# Create one asynchronous Ollama client.
# Ollama communicates with the local Ollama service.
async_client = AsyncClient()


class AskRequest(BaseModel):
    # Text supplied by the API caller.
    query: str


@app.get("/health")
async def health_check():
    # Lightweight application health probe.
    # This does not invoke the LLM.
    return {
        "status": "ok",
        "service": "agentic-rag-engine",
        "model": OLLAMA_MODEL
    }


@app.post("/ask")
async def ask_endpoint(request: AskRequest):
    # Reject empty or whitespace-only queries.
    if not request.query.strip():
        raise HTTPException(
            status_code=400,
            detail="Query string cannot be empty."
        )

    # Send the request to the local Ollama model.
    response = await async_client.chat(
        model=OLLAMA_MODEL,
        messages=[
            {"role": "user", "content": request.query}
        ]
    )

    # Return the original query and generated answer as JSON.
    return {
        "query": request.query,
        "answer": response.message.content
    }


async def stream_generator(query: str):
    # Request streaming generation from the local Ollama model.
    response = await async_client.chat(
        model=OLLAMA_MODEL,
        messages=[
            {"role": "user", "content": query}
        ],
        stream=True
    )

    # Yield generated text chunks as they arrive.
    async for chunk in response:
        content = chunk.message.content
        if content:
            yield content


@app.post("/stream")
async def stream_endpoint(request: AskRequest):
    # Return a streaming HTTP response to the caller.
    return StreamingResponse(
        stream_generator(request.query),
        media_type="text/plain"
    )


Overwriting main.py


--- 
## 5. Streamlit Frontend UI
* **Slide Source:** *Week 3 - Slide 16 ("Minimal Streamlit UI")*
* **Action:** Write `app.py` constructing UI consuming the FastAPI streaming response in real-time.

In [2]:
%%writefile app.py
# Slide Source: Week 3 - Slide 16 ("Minimal Streamlit UI")
# Action: Construct UI consuming the FastAPI streaming response in real-time.

import streamlit as st
import requests

st.set_page_config(page_title="Agentic RAG Control Center", layout="wide")
st.title("Agentic AI & RAG Interface")

query_input = st.text_input("Enter your request or prompt:", placeholder="Ask something...")

if st.button("Submit Query"):
    if not query_input.strip():
        st.warning("Please enter a valid query.")
    else:
        st.subheader("Streaming Response:")
        response_box = st.empty()
        full_response = ""
        
        try:
            url = "http://localhost:8000/stream"
            with requests.post(url, json={"query": query_input}, stream=True) as response:
                if response.status_code == 200:
                    for chunk in response.iter_content(chunk_size=1024, decode_unicode=True):
                        if chunk:
                            full_response += chunk
                            response_box.markdown(full_response + "▌")
                    response_box.markdown(full_response)
                else:
                    st.error(f"Error {response.status_code}: Unable to reach API.")
        except Exception as e:
            st.error(f"Connection error: {str(e)}")

# Run with command: streamlit run app.py

Overwriting app.py


--- 
## 6. Week 3 (Day 2): Testing, Mocks & API Contract Specification
* **Slide Source:** *Week 3 - Slide 28-30 ("Testing & Mocks"), Slide 33-36 ("API Contracts & ADR 0002")*
* **Action:** Unit test pipeline components with `pytest` & `AsyncMock`, and document Architectural Decision Record (ADR 0002).

In [3]:
# Slide Source: Week 3 - Slide 28-30 ("Testing & Mocks")
#
# Action: Test execution components without making real local-model calls.
# AsyncMock simulates the Ollama AsyncClient response.
#
# The test deliberately avoids inference, so it is free and fast.

import pytest
from unittest.mock import AsyncMock
from pydantic import BaseModel


class QuestionRequest(BaseModel):
    id: int
    query: str


class AnswerResponse(BaseModel):
    id: int
    query: str
    answer: str


async def dummy_llm_call(query: str, client) -> str:
    # Call the same method shape used by the production Ollama client.
    response = await client.chat(
        model="gpt-oss:20b",
        messages=[
            {"role": "user", "content": query}
        ]
    )

    # Return the generated message text.
    return response.message.content


@pytest.mark.asyncio
async def test_dummy_llm_call_success():
    # Create a fake asynchronous Ollama client.
    mock_client = AsyncMock()

    # Build a fake response object with the structure expected
    # by dummy_llm_call().
    mock_message = AsyncMock()
    mock_message.content = "Mocked answer payload"

    mock_completion = AsyncMock()
    mock_completion.message = mock_message

    # Configure the fake client's chat() call to return our mock response.
    mock_client.chat.return_value = mock_completion

    # Execute the function under test.
    result = await dummy_llm_call(
        "Test Query",
        mock_client
    )

    # Verify the returned answer.
    assert result == "Mocked answer payload"

    # Verify that the local-model client was called exactly once.
    mock_client.chat.assert_called_once()

    print("Test passed successfully!")


# Run the asynchronous test directly in the notebook.
await test_dummy_llm_call_success()


Test passed successfully!


### ADR 0002: API Interface Contract Locking

**Context:** Standardizing the interface contract for client applications interacting with the RAG microservice.

**Decision:** All standard interactions will adhere strictly to Pydantic JSON validation schemas.

#### Schema Spec: `/v1/ask`
* **Request (POST):** `{"query": "string (min_length: 3)"}`
* **Response (200 OK):** `{"query": "string", "answer": "string"}`
* **Error Response (400 Bad Request):** `{"detail": "Query string cannot be empty."}`